# TCT Setup Control — Application Overview

This notebook explains the architecture, components, and data flow of the **TCT Setup Control** application located in `TCT_app/`.  
It is a reference document — no hardware is required to read it.

---

## Table of Contents
1. [What is TCT?](#1-what-is-tct)
2. [Project structure](#2-project-structure)
3. [Entry point and main window](#3-entry-point-and-main-window)
4. [Hardware abstraction layer](#4-hardware-abstraction-layer)
5. [Device manager and YAML configuration](#5-device-manager-and-yaml-configuration)
6. [Scan controller and state machine](#6-scan-controller-and-state-machine)
7. [GUI panels](#7-gui-panels)
8. [Data pipeline — HDF5 and InfluxDB](#8-data-pipeline)
9. [Waveform analysis](#9-waveform-analysis)
10. [Post-scan analysis panel](#10-post-scan-analysis-panel)
11. [How a scan works end-to-end](#11-end-to-end-scan-flow)
12. [Adding a new hardware backend](#12-adding-a-new-hardware-backend)

---
## 1  What is TCT?

**Transient Current Technique (TCT)** is a standard method for characterising silicon particle detectors.  
A focused laser pulse (typically 660 nm or 1064 nm) is directed onto the sensor, generating electron-hole pairs.  
These carriers drift through the sensor under an applied electric field (bias voltage), producing a transient current signal on the oscilloscope.

Key measurables:

| Quantity | Physical meaning |
|---|---|
| Collected charge (pC) | Efficiency of charge collection (CCE) |
| Drift time (ns) | Carrier velocity → electric field profile |
| Rise time (ns) | Sensor / electronics bandwidth |
| 2-D scan map | Spatial uniformity of charge collection |
| IV curve | Leakage current vs. reverse bias → breakdown voltage |

---
## 2  Project Structure

```
TCT_Setup/
│
├── TCT_app/
│   ├── main.py                    ← entry point (run this)
│   ├── tct_gui.py                 ← TCTMainWindow (assembles all panels)
│   │
│   ├── configs/
│   │   └── devices.yaml           ← hardware backend selection
│   │
│   ├── devices/                   ← hardware abstraction layer
│   │   ├── base.py                  BaseDevice ABC
│   │   ├── motor_base.py            MotorStageBase ABC
│   │   ├── motor_simulated.py       Simulated stage (no hardware)
│   │   ├── motor_pi.py              PI C-863 controller
│   │   ├── motor_grbl.py            GRBL 3D-printer stage (G-code)
│   │   ├── intensity_base.py        IntensityMonitorBase ABC
│   │   ├── intensity_scope_ch.py    Photodiode via oscilloscope channel
│   │   ├── intensity_simulated.py
│   │   ├── oscilloscope.py          Multi-vendor oscilloscope driver
│   │   ├── waveform_generator.py    Pulse/trigger generator
│   │   ├── camera_blackfly.py       FLIR Blackfly camera
│   │   ├── laser_manual.py          Metadata-only laser record
│   │   ├── bias_supply_base.py      BiasSupplyBase ABC
│   │   ├── bias_supply_keithley.py  Keithley 2400/2410/6487 driver
│   │   └── bias_supply_simulated.py
│   │
│   ├── controller/
│   │   ├── device_manager.py        Owns all device instances
│   │   ├── scan_controller.py       Scan loop, Z-focus, voltage scan
│   │   ├── slow_control_manager.py  Polls temperature/humidity sensors
│   │   └── state_machine.py         DISCONNECTED→CONNECTED→RUNNING…
│   │
│   ├── gui/
│   │   ├── motor_panel.py           Stage position + jog controls
│   │   ├── intensity_panel.py       Live photodiode readout
│   │   ├── camera_panel.py          Live camera feed + crosshair
│   │   ├── scope_panel.py           Live waveforms + analysis markers
│   │   ├── laser_panel.py           Laser metadata + trigger settings
│   │   ├── scan_panel.py            Scan config + live inline 2D map
│   │   ├── scan_map_window.py       Detachable full-size 2D map window
│   │   ├── bias_panel.py            Keithley control + IV scan
│   │   ├── monitor_panel.py         Slow-control dashboard
│   │   └── analysis_panel.py        Post-scan HDF5 re-analysis
│   │
│   ├── analysis/
│   │   ├── waveform_analysis.py     Charge, drift time, CFD, rise time
│   │   ├── laser_normalization.py   Normalise DUT charge by reference
│   │   └── efield_analysis.py       V_dep estimation from CCE curve
│   │
│   └── data/
│       ├── hdf5_writer.py           Writes one HDF5 file per run
│       └── influx_writer.py         Optional live InfluxDB sink
│
└── Bilder/                          Miscellaneous images / docs
```

---
## 3  Entry Point and Main Window

```bash
cd TCT_app
python main.py
```

`main.py` sets up Python logging, creates a `QApplication`, and opens `TCTMainWindow`.  
`tct_gui.py / TCTMainWindow` is the central integration point:

```python
class TCTMainWindow(QMainWindow):
    def __init__(self, config_path):
        self._sm      = StateMachine()          # tracks DISCONNECTED → RUNNING …
        self._devices = DeviceManager(config_path)  # owns all hardware objects
        self._writer  = HDF5Writer("runs/run_00001") # per-run data file
        self._scanner = ScanController(devices, sm, writer)

        # Instantiate all GUI panels …
        # Wire panel signals → controller slots …
```

All wiring between GUI signals and hardware calls is in this file.  
Individual panels never talk to hardware directly — they emit signals or call controller methods.

---
## 4  Hardware Abstraction Layer

Every device family has an **Abstract Base Class (ABC)** in `devices/`.  
Concrete backends inherit from the ABC and implement its abstract methods.

### Motor stage (`motor_base.py`)

```python
class MotorStageBase(BaseDevice, ABC):
    @abstractmethod
    def home(self) -> None: ...

    @abstractmethod
    def move_to(self, x_mm, y_mm, z_mm) -> None: ...

    @abstractmethod
    def get_position(self) -> tuple[float, float, float]: ...

    @abstractmethod
    def wait_until_ready(self) -> None: ...

    @abstractmethod
    def stop(self) -> None: ...
```

Available backends:

| YAML key | Class | Communication |
|---|---|---|
| `simulated` | `SimulatedMotorStage` | in-process fake |
| `pi` | `PIMotorStage` | serial RS-232 (PI C-863) |
| `grbl` | `GRBLMotorStage` | serial G-code (3D printer board) |

### Bias supply (`bias_supply_base.py`)

```python
class BiasSupplyBase(BaseDevice, ABC):
    @abstractmethod
    def set_voltage(self, v: float) -> None: ...

    @abstractmethod
    def set_compliance(self, a: float) -> None: ...

    @abstractmethod
    def ramp_to(self, v, step_V, delay_s) -> None: ...

    @abstractmethod
    def read(self) -> BiasReading: ...

    @abstractmethod
    def output_on(self) -> None: ...

    @abstractmethod
    def output_off(self) -> None: ...
```

`BiasReading` is a dataclass with `voltage_V`, `current_A`, `compliance_A`, `output_on`, `compliant`.

The Keithley driver auto-detects the model family from `*IDN?` and selects the correct SCPI command set (2400/2410/2450 or 6487/6517).

---
## 5  Device Manager and YAML Configuration

`controller/device_manager.py` reads `configs/devices.yaml` and instantiates the correct concrete class for each device.  
No GUI code imports a concrete driver — it only holds references to ABCs.

### Example `devices.yaml`

```yaml
motor_stage:
  backend: grbl          # simulated | pi | grbl
  serial_port: COM4
  baudrate: 115200
  velocity_mm_min: 3000

oscilloscope:
  vendor: tektronix      # tektronix | rohde_schwarz | custom_script
  visa_address: "USB0::0x0699::0x0368::XXXXXXX::INSTR"
  simulation: false

bias_supply:
  backend: keithley      # keithley | simulated
  visa_address: "GPIB0::15::INSTR"
  compliance_A: 100e-6   # ALWAYS set a safe limit!
  voltage_range_V: 1100

intensity_monitor:
  backend: scope_channel
  channel: 1             # oscilloscope channel for reference photodiode

camera:
  simulation: true       # set false + serial_number for real Blackfly

waveform_generator:
  visa_address: "USB0::..."
  frequency_hz: 1000
  pulse_width_s: 100e-9
  amplitude_V: 3.3
  simulation: false

slow_control:
  poll_interval_s: 10
  channels: []           # add SNMP/Modbus channels here

influx:                  # optional — leave empty to disable
  url: "http://localhost:8086"
  token: "..."
  org: "myorg"
  bucket: "tct"
```

### Backend registry pattern

```python
MOTOR_BACKENDS = {
    "simulated": SimulatedMotorStage,
    "pi":        PIMotorStage,
    "grbl":      GRBLMotorStage,
}
# Looked up at startup:
motor_cls = MOTOR_BACKENDS[cfg["motor_stage"]["backend"]]
```

To add a new motor backend: create the class, add it to `MOTOR_BACKENDS`, update `devices.yaml`.

---
## 6  Scan Controller and State Machine

### State machine (`controller/state_machine.py`)

```
DISCONNECTED ──connect──▶ CONNECTED ──start──▶ RUNNING
                                                  │  ▲
                                                pause │ resume
                                                  ▼  │
                                               PAUSED
                                                  │
                                           abort/finish/error
                                                  ▼
                                     ABORTED / FINISHED / ERROR
                                                  │
                                           ──reconnect──▶ CONNECTED
```

The state machine guards against starting a scan while disconnected, starting two scans simultaneously, etc.

### Scan controller (`controller/scan_controller.py`)

Three scan modes:

| Method | Config dataclass | What it does |
|---|---|---|
| `start(cfg)` | `ScanConfig` | Full 2-D XY scan at fixed Z and bias |
| `start_z_focus_scan(cfg)` | `ZFocusScanConfig` | Sweeps Z at fixed XY, finds amplitude peak |
| `start_voltage_scan(cfg)` | `VoltageScanConfig` | Steps bias voltage, acquires waveforms per step |

All three run in a **daemon background thread** so the GUI stays responsive.  
Communication back to the GUI uses callback functions (set as attributes of `ScanController`):  
`on_point_done`, `on_progress`, `on_finished`, `on_error`, `on_vscan_point`.

### ScanConfig fields

```python
@dataclass
class ScanConfig:
    x_start_mm: float = -1.0
    x_stop_mm:  float =  1.0
    x_step_mm:  float =  0.1
    y_start_mm: float = -1.0
    y_stop_mm:  float =  1.0
    y_step_mm:  float =  0.1
    z_mm:       float =  0.0     # fixed Z during scan
    n_averages: int   =  1       # waveforms averaged per point
    settle_time_s: float = 0.05  # wait after move before acquisition
```

The scan uses a **boustrophedon** (snake) path to minimise stage travel distance.

In [ ]:
# Visualise the boustrophedon scan path
import numpy as np
import matplotlib.pyplot as plt

x_start, x_stop, x_step = -1.0, 1.0, 0.4
y_start, y_stop, y_step = -1.0, 1.0, 0.4

xs = np.arange(x_start, x_stop + x_step / 2, x_step)
ys = np.arange(y_start, y_stop + y_step / 2, y_step)

path_x, path_y = [], []
for i, x in enumerate(xs):
    row = ys if i % 2 == 0 else ys[::-1]
    for y in row:
        path_x.append(x)
        path_y.append(y)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(path_x, path_y, '-o', markersize=6, linewidth=1)
for i, (x, y) in enumerate(zip(path_x, path_y)):
    ax.annotate(str(i), (x, y), fontsize=7, ha='center', va='bottom')
ax.set_xlabel('X (mm)'); ax.set_ylabel('Y (mm)')
ax.set_title('Boustrophedon (snake) scan path')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7  GUI Panels

All panels are `QWidget` subclasses.  They communicate with the rest of the application **only** via Qt signals and the controller callbacks — no direct hardware calls from GUI code.

| Tab | Panel class | Key features |
|---|---|---|
| Motor / Intensity | `MotorPanel` + `IntensityPanel` | XYZ position, jog, home, set-as-start; live photodiode reading |
| Camera | `CameraPanel` | Live FLIR Blackfly feed at 10 fps, crosshair overlay, save frame |
| Oscilloscope | `ScopePanel` | Single + live waveform display with analysis markers (onset, CFD, trailing, integration window), CSV export |
| Laser / Trigger | `LaserPanel` | Laser metadata, waveform generator output control |
| Scan | `ScanPanel` | Scan grid parameters, ETA, live inline 2D map, Z-focus sub-panel, save/load params, **Open 2D Map Window** button |
| Scan Map (detached) | `ScanMapWindow` | Full-size 2D map, quantity selector, auto-levels, PNG + CSV export |
| Bias Supply | `BiasPanel` | Voltage ramp, compliance, live V/I readout, compliance trip alarm, IV scan (background thread), bias+waveform scan |
| Monitor | `MonitorPanel` | Slow-control channel table (colour-coded alarm), history plot |
| Analysis | `AnalysisPanel` | Load HDF5 file, re-plot 2D map, CCE vs. bias curve, V_dep estimation, CSV export |

### ROI in 2D maps

Both `ScanPanel` and `AnalysisPanel` use `pg.ImageView` which has a built-in `ROI` (region-of-interest selector).  
Rotation is **disabled** — only axis-aligned X/Y resizing is allowed, which is the only physically meaningful operation for a rectangular scan grid.  
To resize along X and Y independently, drag the side handles of the ROI rectangle.

---
## 8  Data Pipeline

### HDF5 writer (`data/hdf5_writer.py`)

One HDF5 file is created per run in `runs/run_NNNNN/`.  
The file structure:

```
run_00001.h5
├── points/
│   ├── x_mm        [N]   stage X position
│   ├── y_mm        [N]   stage Y position
│   ├── z_mm        [N]   stage Z position
│   ├── timestamp   [N]   Unix timestamp
│   ├── dut_charge_pC   [N]
│   ├── dut_amplitude_V [N]
│   ├── ref_amplitude_V [N]
│   ├── baseline_rms_V  [N]
│   ├── drift_time_s    [N]
│   ├── rise_time_s     [N]
│   └── cfd_time_s      [N]
└── waveforms/
    ├── time_axis   [N, M]   (M samples per waveform)
    ├── dut         [N, M]
    └── ref         [N, M]
```

### InfluxDB sink (`data/influx_writer.py`)

If `influx:` is configured in `devices.yaml`, slow-control readings are pushed to InfluxDB in real time for Grafana dashboarding.  
If `influx:` is absent or empty, the `InfluxWriter` is a no-op and no network calls are made.

### Reading data back in Python

```python
import h5py, numpy as np

with h5py.File('runs/run_00001/run_00001.h5', 'r') as f:
    x   = f['points/x_mm'][:]
    y   = f['points/y_mm'][:]
    chg = f['points/dut_charge_pC'][:]

# Pivot to 2D map
xs, ys = np.unique(x), np.unique(y)
arr = np.zeros((len(xs), len(ys)))
xi  = {v: i for i, v in enumerate(xs)}
yi  = {v: i for i, v in enumerate(ys)}
for xi_, yi_, c in zip(x, y, chg):
    arr[xi[xi_], yi[yi_]] = c
```

---
## 9  Waveform Analysis

`analysis/waveform_analysis.py` contains `analyse_waveform(time_s, voltage_V)` which returns a `WaveformResult` dataclass.

### Analysis steps

1. **Baseline**: mean of the first `baseline_samples` (default 20) points before the signal
2. **Baseline subtraction**: `corrected = voltage_V − baseline`
3. **Amplitude**: `max(|corrected|)`
4. **Charge**: integrate corrected waveform over the window `(20 ns, 150 ns)` divided by termination resistance (50 Ω by default)
5. **CFD time**: linear interpolation where `corrected` first crosses `cfd_fraction × amplitude` (default 30 %)
6. **Rise time**: time between 10 % and 90 % of peak amplitude
7. **Onset time**: first crossing of `onset_threshold × amplitude` on the leading edge
8. **Trailing time**: last crossing of `onset_threshold × amplitude` on the trailing edge
9. **Drift time**: `trailing_time − onset_time` = carrier transit time through the sensor

In [ ]:
# Demonstrate waveform analysis on a synthetic TCT pulse
import numpy as np
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'TCT_app'))

from analysis.waveform_analysis import analyse_waveform

# Synthetic TCT waveform: trapezoidal pulse + noise
t = np.linspace(0, 250e-9, 2500)   # 250 ns window
onset_t, trailing_t = 40e-9, 180e-9
v = np.where(
    (t >= onset_t) & (t <= trailing_t),
    0.05 * np.sin(np.pi * (t - onset_t) / (trailing_t - onset_t)),
    0.0
) + np.random.normal(0, 5e-4, len(t))

result = analyse_waveform(t, v)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t * 1e9, v * 1e3, 'k', linewidth=0.8, label='Waveform')
ax.axvspan(20, 150, alpha=0.08, color='blue', label='Integration window')
if result.onset_time_s:    ax.axvline(result.onset_time_s * 1e9, color='green', label='Onset', linestyle='--')
if result.trailing_time_s: ax.axvline(result.trailing_time_s * 1e9, color='red', label='Trailing', linestyle='--')
if result.cfd_time_s:      ax.axvline(result.cfd_time_s * 1e9, color='gold', label='CFD (30%)', linestyle=':')
ax.set_xlabel('Time (ns)'); ax.set_ylabel('Voltage (mV)')
ax.set_title('Synthetic TCT waveform — analysis markers')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Amplitude  : {result.amplitude_V * 1e3:.2f} mV")
print(f"Charge     : {result.charge_pC:.3f} pC")
print(f"Drift time : {result.drift_time_s * 1e9:.1f} ns" if result.drift_time_s else "Drift time : N/A")
print(f"CFD time   : {result.cfd_time_s * 1e9:.1f} ns" if result.cfd_time_s else "CFD time   : N/A")
print(f"Rise time  : {result.rise_time_s * 1e9:.1f} ns" if result.rise_time_s else "Rise time  : N/A")

---
## 10  Post-scan Analysis Panel

The **Analysis** tab loads a completed HDF5 run file and provides:

### Tab A — 2D map re-plot
- Choose any stored quantity (`dut_charge_pC`, `dut_amplitude_V`, `drift_time_s`, …)
- `pg.ImageView` with ROI for zooming (rotation disabled, X/Y resize via corner handles)
- Axis scale set to mm so the ROI position reads in physical units
- Export to CSV

### Tab B — CCE vs. bias
- Load a **voltage scan** HDF5 file (produced by the Bias + Waveform Scan)
- CCE = `dut_charge_pC / Q_ref`  where `Q_ref` is the charge at full depletion (user-set)
- Leakage current overlay (scaled to CCE axis)
- `V_dep` estimation from `analysis/efield_analysis.py`

---
## 11  End-to-End Scan Flow

```
User presses "▶ Start Scan"
        │
        ▼
ScanPanel emits start_requested(ScanConfig)
        │
        ▼
TCTMainWindow._start_scan(cfg)
    • checks StateMachine.can(RUNNING)
    • creates next run HDF5Writer
    • calls scanner.start(cfg)
        │
        ▼
ScanController._run() — background thread
    for each point in snake path:
        1. motor.move_to(x, y, z)          ← MotorStageBase
        2. motor.wait_until_ready()
        3. time.sleep(settle_time_s)
        4. scope.read_channel(1) → ref waveform
        5. scope.read_channel(2) → DUT waveform
        6. analyse_waveform(DUT) → WaveformResult
        7. normalise(DUT, ref)  → normalised charge
        8. bias_supply.read()  → check compliance trip
        9. writer.save_point(result)
       10. on_point_done(result) → ScanPanel.on_point_done()
           → updates inline 2D map
           → updates detached ScanMapWindow (if open)
       11. on_progress(done, total) → ScanPanel.on_progress()
    ─────────────────────────────────────────────────────
    on_finished() → TCTMainWindow._on_scan_finished()
    writer.close() → HDF5 file flushed and closed
    waveform_generator.output_off()
```

### Safety interlock

After every acquisition point the scan thread reads `bias_supply.read()`.  
If `reading.compliant` is `True` (compliance trip = sensor breakdown), the scan is aborted immediately and the bias is ramped to 0 V.

---
## 12  Adding a New Hardware Backend

### Example: new motor stage

**Step 1** — Create `devices/motor_newport.py`:

```python
from devices.motor_base import MotorStageBase

class NewportMotorStage(MotorStageBase):
    def connect(self) -> None: ...
    def disconnect(self) -> None: ...
    def home(self) -> None: ...
    def move_to(self, x_mm, y_mm, z_mm) -> None: ...
    def get_position(self) -> tuple[float, float, float]: ...
    def wait_until_ready(self) -> None: ...
    def stop(self) -> None: ...
```

**Step 2** — Register in `controller/device_manager.py`:

```python
from devices.motor_newport import NewportMotorStage

MOTOR_BACKENDS = {
    ...
    "newport": NewportMotorStage,
}
# Add instantiation branch (if non-trivial constructor params needed):
elif motor_backend == "newport":
    self.motor = NewportMotorStage(port=motor_cfg.get("serial_port", "COM5"))
```

**Step 3** — Update `configs/devices.yaml`:

```yaml
motor_stage:
  backend: newport
  serial_port: COM5
```

No GUI code changes needed — the rest of the application uses `MotorStageBase`.

---

### Quick-start checklist for a new measurement session

1. Edit `configs/devices.yaml` to match the hardware connected (backend names, VISA addresses, serial ports)
2. Launch: `python main.py`
3. Click **Connect All** — check the status bar for any failed connections
4. Use the **Oscilloscope** tab to verify waveform quality at one position
5. Run a **Z-focus** scan from the **Scan** tab to find the laser focal plane
6. In the **Bias Supply** tab, apply the desired reverse bias (set compliance first!)
7. Configure the XY scan grid in the **Scan** tab and press **▶ Start Scan**
8. Monitor progress in the inline 2D map or open **⊞ Open 2D Map Window** for a full-size view
9. After the scan, load the HDF5 file in the **Analysis** tab for CCE maps and bias scans